# Lab 6: Нэгдүгээр эрэмбийн логик (First-Order Logic)
Prolog ба pyDatalog ашиглан гэр бүлийн мэдлэгийн сан байгуулах

## Алхам 1: Орчин бэлтгэх

pyDatalog санг ачааллаж, `clear()` дуудна. Энэ мөр **заавал** эхний cell-д байх ёстой — учир нь cell-ийг дахин ажиллуулахад өмнөх баримтууд хуримтлагдаж давхардахаас сэргийлнэ.

In [1]:
from pyDatalog import pyDatalog
pyDatalog.clear()

### Жижиг жишээ: pyDatalog зөв ажиллаж байгааг шалгах

`create_terms` нь нэрсийг (predicate, variable) Python орчинд танигдах болгож зарладаг — JavaScript-ийн `var x` шиг. `+bird('tweety')` гэдгийн `+` тэмдэг **баримт нэмэх** гэсэн үг. `flies(X) <= bird(X)` бол **дүрэм**: "аливаа X шувуу бол X нисдэг". `print(flies(X))` нь query — хоёр утга (tweety, sparrow) буцаах ёстой.

In [2]:
pyDatalog.create_terms('bird, flies, X')
+bird('tweety')
+bird('sparrow')
flies(X) <= bird(X)
print(flies(X))

X      
-------
sparrow
tweety 


Жишээг шалгасны дараа `clear()` дуудаж bird/flies баримтыг арилгана, дараа нь гэр бүлийн лабд хэрэглэх бүх предикат, хувьсагчийн нэрийг нэг дор зарлана.

In [3]:
pyDatalog.clear()

_ = pyDatalog.create_terms(
    'male, female, parent, person, has_child, childless, '
    'father, mother, son, daughter, grandparent, sibling, '
    'brother, sister, uncle, aunt, cousin, ancestor, descendant, '
    'great_grandparent, '
    'X, Y, Z, P, Q'
)

## Алхам 2: Гэр бүлийн мэдлэгийн сан (баримт)

15 хүнтэй, дөрвөн үетэй гэр бүл. Нэрийг латин жижиг үсгээр бичнэ (Монгол нэрийг Prolog/pyDatalog-д шууд ашиглахад бэрхшээлтэй).

Гурван предикат бичнэ:
- `male(X)` — X эрэгтэй (8 баримт)
- `female(X)` — X эмэгтэй (7 баримт)
- `parent(P, C)` — P нь C-ийн эцэг эх (18 баримт). **Эхний аргумент = эцэг эх, хоёр дахь = хүүхэд.**

**Зөвлөгөө:** `parent` баримтыг 18 мөр гараар бичихийн оронд, эцэг эх хосын жагсаалт ба хүүхдийн жагсаалтаар давталт хийнэ. Ингэснээр гараар бичих алдаа гарахгүй, мөн эцэг ба эх хоёул тусдаа баримт болж орно.

In [4]:
males = ['bataa', 'bold', 'tomor', 'dorj', 'zolboo', 'munkh', 'temuulen', 'bat']
females = ['saraa', 'oyuna', 'tsetseg', 'ganaa', 'anar', 'naraa', 'enkhee']

for m in males:
    +male(m)
for f in females:
    +female(f)

print(len(males), 'male facts,', len(females), 'female facts')

8 male facts, 7 female facts


In [5]:
# (эцэг эхүүд, хүүхдүүд) хосуудаар parent баримт нэмнэ.
# Нэг гэр бүлд нэг давталт -> эцэг ба эх тус бүрээс хүүхэд тус бүр рүү parent үүснэ.
families = [
    (['bataa', 'saraa'], ['bold', 'oyuna', 'tomor']),      # Yе1 -> Yе2
    (['bold', 'tsetseg'], ['anar', 'zolboo']),              # Yе2 -> Yе3
    (['dorj', 'oyuna'], ['munkh', 'naraa']),                # Yе2 -> Yе3
    (['tomor', 'ganaa'], ['enkhee']),                       # Yе2 -> Yе3
    (['bat', 'anar'], ['temuulen']),                        # Yе3 -> Yе4
]

for parents, children in families:
    for p in parents:
        for c in children:
            +parent(p, c)

print('parent facts added')

parent facts added


`pyDatalog.ask("...")` нь үр дүн олдохгүй бол `None` буцаадаг (хоосон жагсаалт биш). Тиймээс асуултын хариултын тоог найдвартай тоолох `count(q)` функц бичнэ — `None` үед 0 буцаана. Үүнийг цаашдын бүх шалгалтад ашиглана.

In [6]:
def count(query_string):
    result = pyDatalog.ask(query_string)
    return len(result.answers) if result else 0

print(count("parent(X, Y)"))
print(count("male(X)"))
print(count("female(X)"))

18
8
7


**Шалгах:** `count("parent(X, Y)")` нь 18, `count("male(X)")` нь 8, `count("female(X)")` нь 7 байх ёстой. `parent('bataa', X)` асуулт bold, oyuna, tomor гурвыг буцаана.

In [7]:
assert count("parent(X, Y)") == 18
assert count("male(X)") == 8
assert count("female(X)") == 7

print(parent('bataa', X))

X    
-----
oyuna
tomor
bold 


## Алхам 3: Энгийн дүрэм

Баримтаас шууд гаргаж болох дөрвөн дүрэм:

| Дүрэм | Утга |
|---|---|
| `father(X, Y)` | X нь Y-ийн эцэг: X нь Y-ийн эцэг эх бөгөөд эрэгтэй |
| `mother(X, Y)` | X нь Y-ийн эх: X нь Y-ийн эцэг эх бөгөөд эмэгтэй |
| `son(X, Y)` | X нь Y-ийн хүү: Y нь X-ийн эцэг эх бөгөөд X эрэгтэй |
| `daughter(X, Y)` | X нь Y-ийн охин: Y нь X-ийн эцэг эх бөгөөд X эмэгтэй |

Мөн `person(X)` дүрэм: эрэгтэй **эсвэл** эмэгтэй хэн бүхэн person. Нэг предикатад хоёр тусдаа дүрэм бичих нь логикийн "эсвэл" (∨)-ийг илэрхийлнэ — аль нэг нь үнэн бол хангалттай.

FOL томъёо: ∀x ∀y (parent(x, y) ∧ male(x) → father(x, y))

In [8]:
# father(X, Y): X нь Y-ийн эцэг эх, мөн X эрэгтэй
father(X, Y) <= parent(X, Y) & male(X)

# mother(X, Y): X нь Y-ийн эцэг эх, мөн X эмэгтэй
mother(X, Y) <= parent(X, Y) & female(X)

# son(X, Y): Y нь X-ийн эцэг эх, мөн X эрэгтэй
son(X, Y) <= parent(Y, X) & male(X)

# daughter(X, Y): Y нь X-ийн эцэг эх, мөн X эмэгтэй
daughter(X, Y) <= parent(Y, X) & female(X)

# person(X): эрэгтэй эсвэл эмэгтэй хэн бүхэн -> хоёр тусдаа дүрэм = "эсвэл"
person(X) <= male(X)
person(X) <= female(X)

print('rules defined')

rules defined


**Шалгах:** `father(X, 'anar')` нь bold, `mother(X, 'bold')` нь saraa байна. `son(X, 'oyuna')` нь munkh, `daughter(X, 'oyuna')` нь naraa. `count("father(X, Y)")` ба `count("mother(X, Y)")` тус бүр 9. `count("person(X)")` нь 15.

In [9]:
print(father(X, 'anar'))
print(mother(X, 'bold'))
print(son(X, 'oyuna'))
print(daughter(X, 'oyuna'))

assert count("father(X, Y)") == 9
assert count("mother(X, Y)") == 9
assert count("person(X)") == 15
print('all checks passed')

X   
----
bold
X    
-----
saraa
X    
-----
munkh
X    
-----
naraa
all checks passed


## Алхам 4: Өвөө эмээ ба ах дүү

| Дүрэм | Утга |
|---|---|
| `grandparent(X, Y)` | X нь Y-ийн эцэг эхийн эцэг эх. Дунд нь Z хүн: X нь Z-ийн эцэг эх, Z нь Y-ийн эцэг эх |
| `sibling(X, Y)` | X ба Y нэг эцэг эхтэй, мөн X ба Y өөр хүн |

**Анхаар:** `sibling` дүрэмд `X != Y` нөхцөл заавал байх ёстой — үгүй бол хүн бүр өөрийнхөө ах дүү болно (учир нь X өөрийн эцэг эхийн хүүхэд мөн).

`grandparent` доторх `Z` бол **∃ (оршин байх)** хувьсагч — ямар нэг Z олдвол хангалттай, бүх Z-д биш. Харин `X, Y` нь дүрмийн толгойд байгаа тул дуудагч тал тодорхойлно (∀ шиг ойлгоно).

FOL: ∃z (parent(x, z) ∧ parent(z, y)) → grandparent(x, y)

In [10]:
# grandparent(X, Y): X -> Z -> Y гэсэн хоёр parent холбоос дамжина
grandparent(X, Y) <= parent(X, Z) & parent(Z, Y)

# sibling(X, Y): X, Y нэг эцэг эхтэй (Z), мөн X өөр хүн (X != Y тэмдэг заавал)
sibling(X, Y) <= parent(Z, X) & parent(Z, Y) & (X != Y)

print('rules defined')

rules defined


**Шалгах:** `grandparent(X, 'temuulen')` нь bold, tsetseg. `count("grandparent(X, Y)")` нь 12. `sibling(X, 'munkh')` нь naraa. `sibling(X, 'bold')` нь oyuna, tomor. `count("sibling(X, Y)")` нь 10 (эрэмбэлсэн хос тул жишээ нь (bold,oyuna) ба (oyuna,bold) хоёул тоологдоно).

In [11]:
print(grandparent(X, 'temuulen'))
print(sibling(X, 'munkh'))
print(sibling(X, 'bold'))

assert count("grandparent(X, Y)") == 12
assert count("sibling(X, Y)") == 10
print('all checks passed')

X      
-------
tsetseg
bold   
X    
-----
naraa
X    
-----
tomor
oyuna
all checks passed


### Туршилт: `X != Y` нөхцөлийг хасвал юу болох вэ?

Доорх cell-д зөвхөн туршилтын зорилгоор `X != Y`-гүй `sibling_test` дүрэм үүсгэнэ (жинхэнэ `sibling`-ийг өөрчлөхгүй).

In [12]:
_ = pyDatalog.create_terms('sibling_test')
sibling_test(X, Y) <= parent(Z, X) & parent(Z, Y)  # X != Y байхгүй

print('sibling(X,Y):', count("sibling(X, Y)"))
print('sibling_test(X,Y):', count("sibling_test(X, Y)"))
print('зөрүү:', count("sibling_test(X, Y)") - count("sibling(X, Y)"))

sibling(X,Y): 10
sibling_test(X,Y): 19
зөрүү: 9


**Яагаад яг 9 вэ?** Datalog-ийн дүрэм нь **олонлог** (set) үүсгэдэг — `sibling_test(X, X)` үнэн гэдгийг хичнээн удаа (хэдэн Z-ээр) баталсан ч нэг л удаа тоологдоно, давхардахгүй. Тиймээс нэмэгдсэн хос бүр нь `(X, X)` хэлбэртэй **өөрийгөө ах дүү гэж тооцсон** тохиолдол бөгөөд ядаж нэг бичигдсэн эцэг эхтэй хүн тутамд яг нэг удаа гарна.

Манай 15 хүнээс **6 хүн**д (bataa, saraa, tsetseg, dorj, ganaa, bat) мэдлэгийн санд бичигдсэн эцэг эх алга (тэд гэр бүлийн модны үндэс буюу гаднаас гэрлэж орсон хүмүүс). Үлдсэн **15 − 6 = 9** хүн тус бүрд ядаж нэг `parent(Z, X)` баримт байгаа тул `(X, X)` хос нэг удаа үүснэ → яг 9 нэмэгдэнэ.

## Алхам 5: Ураг төрлийн дүрмүүд

Өмнөх дүрмүүдээ дахин ашиглана (`sibling`-ийг шууд ашиглаж, `parent`-ээс дахин эхлэхгүй):

| Дүрэм | Утга |
|---|---|
| `brother(X, Y)` | `sibling(X, Y)` ба X эрэгтэй |
| `sister(X, Y)` | `sibling(X, Y)` ба X эмэгтэй |
| `uncle(X, Y)` | Y-ийн эцэг эхийн эрэгтэй ах дүү |
| `aunt(X, Y)` | Y-ийн эцэг эхийн эмэгтэй ах дүү |
| `cousin(X, Y)` | X, Y-ийн эцэг эхүүд ах дүү |

**Анхаар:** энд `uncle`/`aunt` зөвхөн **цусан төрлийн** авга — авга эгчийн нөхөр (жишээ нь dorj) энэ дүрмээр uncle болохгүй (цаашид Нэмэлт даалгаварт засна).

In [13]:
# brother/sister: sibling дээр хүйс нэмнэ
brother(X, Y) <= sibling(X, Y) & male(X)
sister(X, Y) <= sibling(X, Y) & female(X)

# uncle/aunt: X нь Y-ийн эцэг эх (Z)-ийн ах дүү
uncle(X, Y) <= sibling(X, Z) & parent(Z, Y) & male(X)
aunt(X, Y) <= sibling(X, Z) & parent(Z, Y) & female(X)

# cousin: X, Y-ийн эцэг эхүүд (P, Q) хоорондоо ах дүү
cousin(X, Y) <= parent(P, X) & parent(Q, Y) & sibling(P, Q)

print('rules defined')

rules defined


**Шалгах:** `brother(X, 'oyuna')` нь bold, tomor. `sister(X, 'zolboo')` нь anar. `uncle(X, 'anar')` нь tomor, `aunt(X, 'anar')` нь oyuna. `uncle(X, 'temuulen')` нь zolboo. `cousin(X, 'enkhee')` нь anar, munkh, naraa, zolboo (4 хүн). `cousin(X, 'anar')` нь enkhee, munkh, naraa. `count("cousin(X, Y)")` нь 16.

In [14]:
print(brother(X, 'oyuna'))
print(sister(X, 'zolboo'))
print(uncle(X, 'anar'))
print(aunt(X, 'anar'))
print(uncle(X, 'temuulen'))
print(cousin(X, 'enkhee'))
print(cousin(X, 'anar'))

assert count("cousin(X, Y)") == 16
print('all checks passed')

X    
-----
tomor
bold 
X   
----
anar
X    
-----
tomor
X    
-----
oyuna
X     
------
zolboo
X     
------
naraa 
munkh 
zolboo
anar  
X     
------
enkhee
naraa 
munkh 
all checks passed


## Алхам 6: Рекурсив дүрэм

`ancestor`-ийг "хэдэн үе дээш" гэж урьдчилан мэдэх боломжгүй тул **хоёр дүрмээр** тодорхойлно:

- **Суурь дүрэм:** X нь Y-ийн шууд эцэг эх бол X нь Y-ийн өвөг.
- **Рекурсив дүрэм:** X нь Z-ийн шууд эцэг эх, мөн Z нь (аль хэдийн тодорхойлогдсон) Y-ийн өвөг бол X нь Y-ийн өвөг.

`descendant(X, Y)` нь зүгээр `ancestor`-ийг эсрэг чиглэлд хэрэглэнэ.

**Зөвлөгөө:** рекурсив дүрэмд `parent`-ийг эхэнд, `ancestor`-ыг төгсгөлд бич (`parent(X, Z) & ancestor(Z, Y)`, эсрэгээр биш). Prolog дээр энэ дараалал хайлтыг төгсгөлгүй гогцоонд унахаас хамгаална (Алхам 10-ыг үз).

**Гараар шалгах хэлхээ** — `ancestor('bataa', 'temuulen')` үнэн болохыг дөрвөн хүний (bataa, bold, anar, temuulen) хэлхээгээр батлая:

1. `parent(bataa, bold)` → суурь дүрмээр `ancestor(bataa, bold)`
2. `parent(bold, anar)` → суурь дүрмээр `ancestor(bold, anar)`
3. `parent(anar, temuulen)` → суурь дүрмээр `ancestor(anar, temuulen)`
4. `parent(bold, anar)` ба алхам 3-ын `ancestor(anar, temuulen)` → рекурсив дүрмээр `ancestor(bold, temuulen)`
5. `parent(bataa, bold)` ба алхам 4-ийн `ancestor(bold, temuulen)` → рекурсив дүрмээр `ancestor(bataa, temuulen)` ✓

In [15]:
# Суурь дүрэм: шууд эцэг эх -> өвөг
ancestor(X, Y) <= parent(X, Y)

# Рекурсив дүрэм: parent-ийг эхэнд, ancestor-ыг төгсгөлд (Prolog-д гогцоо үүсгэхгүйн тулд)
ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y)

# descendant: ancestor-ийг эсрэг чиглэлд
descendant(X, Y) <= ancestor(Y, X)

print('rules defined')

rules defined


**Шалгах:** `ancestor(X, 'temuulen')` нь 6 хүн: anar, bat, bataa, bold, saraa, tsetseg. `descendant(X, 'bataa')` нь 9 хүн. `ancestor(X, 'anar')` нь bataa, bold, saraa, tsetseg. `count("ancestor(X, Y)")` нь 32.

In [16]:
print(ancestor(X, 'temuulen'))
print(descendant(X, 'bataa'))
print(ancestor(X, 'anar'))

assert count("ancestor(X, Y)") == 32
print('all checks passed')

X      
-------
bold   
tsetseg
bataa  
saraa  
anar   
bat    
X       
--------
bold    
tomor   
oyuna   
anar    
zolboo  
temuulen
enkhee  
munkh   
naraa   
X      
-------
bataa  
saraa  
bold   
tsetseg
all checks passed


## Алхам 7: Асуулт ба үгүйсгэл

Хоёр дүрэм нэмнэ:
- `has_child(X)`: X нь ямар нэг хүний эцэг эх (хувьсагч Y зөвхөн бие дээр, head-д ороогүй тул ∃-аар ойлгоно).
- `childless(X)`: X нь person бөгөөд `has_child` биш.

**Яагаад эхлээд `person(X)`, дараа нь үгүйсгэл вэ? (unsafe negation)** Үгүйсгэл `~has_child(X)` гэдэг нь "X-ийн утга мэдэгдэхгүй үед X-д тохирох бүх боломжит утгыг хайна" гэсэн үг биш — Datalog-д хувьсагч **эхлээд өөр эерэг нөхцөлөөр холбогдсон байх ёстой** (жишээ нь `person(X)`), дараа нь л түүн дээр үгүйсгэл хэрэглэнэ. Хэрэв эхлээд `~has_child(X)` гэж бичвэл X ямар ч утгатай болохыг мэдэхгүй тул хайлт тодорхойгүй (unsafe) болно.

Мөн яагаад `~parent(X, Y)`-г шууд ашиглахгүй вэ? Учир нь бид "хүүхэдгүй" гэдгийг мэдэхийг хүсч байгаа — өөрөөр хэлбэл "ямар ч Y-д parent(X, Y) биш" гэдгийг. Энэ бол **бүх Y-д үгүйсгэл** (∀y ¬parent(x,y)), харин Datalog-ийн нэг хувьсагчтай энгийн үгүйсгэл үүнийг шууд илэрхийлж чадахгүй тул эхлээд `has_child(X)`-ийг тусад нь тодорхойлж, дараа нь түүнийг нь ∀-р бодит байдлаар үгүйсгэнэ.

In [17]:
# has_child(X): X нь Y-ийн эцэг эх болох ямар нэг Y оршин байна (Y нь бие дотор л)
has_child(X) <= parent(X, Y)

# childless(X): эхлээд person(X)-ээр X-ийг холбож, дараа нь has_child биш гэж шалгана
childless(X) <= person(X) & ~has_child(X)

# great_grandparent(X, Y): X нь Z-ийн эцэг эх, Z нь Y-ийн өвөө эмээ
great_grandparent(X, Y) <= parent(X, Z) & grandparent(Z, Y)

print('rules defined')

rules defined


**Шалгах:** `count("childless(X)")` нь 5 (enkhee, munkh, naraa, temuulen, zolboo). `childless('bataa')` худал, `childless('temuulen')` үнэн. `great_grandparent(X, 'temuulen')` нь bataa, saraa.

In [18]:
print(childless(X))
print('bataa childless:', count("childless('bataa')") > 0)
print('temuulen childless:', count("childless('temuulen')") > 0)
print(great_grandparent(X, 'temuulen'))

assert count("childless(X)") == 5
print('all checks passed')

X       
--------
enkhee  
naraa   
temuulen
munkh   
zolboo  
bataa childless: False
temuulen childless: True
X    
-----
saraa
bataa
all checks passed


### Асуултын хүснэгт

Доорх 8 асуултыг мэдлэгийн санд тавьж, хариултыг бичнэ. Хариу бүрийг дээрх гэр бүлийн хүснэгтээс гараар шалгаарай.

In [19]:
# Асуулт 5, 8-д хэрэгтэй туслах предикатууд: "X-д ядаж нэг Y байна уу" гэдгийг
# тусдаа нэрт болгож, дараа нь X-ээр нь давхардалгүй жагсаая.
_ = pyDatalog.create_terms('has_cousin, is_grandparent')

has_cousin(X) <= cousin(X, Y)
is_grandparent(X) <= grandparent(X, Y)

print('helper predicates defined')

helper predicates defined


In [20]:
print('1. Болдын хүүхдүүд хэн бэ?')
print(parent('bold', X))

print('2. Temuulen-ийн өвөө эмээ хэн бэ?')
print(grandparent(X, 'temuulen'))

print('3. Munkh-ийн авга ах нар хэн бэ?')
print(uncle(X, 'munkh'))

print('4. Enkhee-ийн үеэлүүд хэн бэ?')
print(cousin(X, 'enkhee'))

print('5. Үеэлтэй хүмүүс хэд вэ, хэн бэ?')
print('тоо:', count("has_cousin(X)"))
print(has_cousin(X))

print('6. Хүүхэдгүй хүмүүс хэн бэ?')
print(childless(X))

print('7. Temuulen-ийн эцэг эхийн өвөө эмээ хэн бэ? (great_grandparent)')
print(great_grandparent(X, 'temuulen'))

print('8. Ач, зээтэй хүмүүс хэн бэ? (grandparent-ийг ашигла)')
print(is_grandparent(X))

1. Болдын хүүхдүүд хэн бэ?
X     
------
zolboo
anar  
2. Temuulen-ийн өвөө эмээ хэн бэ?
X      
-------
tsetseg
bold   
3. Munkh-ийн авга ах нар хэн бэ?
X    
-----
tomor
bold 
4. Enkhee-ийн үеэлүүд хэн бэ?
X     
------
naraa 
munkh 
zolboo
anar  
5. Үеэлтэй хүмүүс хэд вэ, хэн бэ?
тоо: 5
X     
------
enkhee
munkh 
naraa 
anar  
zolboo
6. Хүүхэдгүй хүмүүс хэн бэ?
X       
--------
enkhee  
naraa   
temuulen
munkh   
zolboo  
7. Temuulen-ийн эцэг эхийн өвөө эмээ хэн бэ? (great_grandparent)
X    
-----
saraa
bataa
8. Ач, зээтэй хүмүүс хэн бэ? (grandparent-ийг ашигла)
X      
-------
tsetseg
bold   
saraa  
bataa  


## Алхам 8: Prolog руу хөрвүүлэх

pyDatalog-ийн бичиглэлийг Prolog руу хөрвүүлэхдээ дараах дүрмийг баримтална (4.5-ын хүснэгт):

| pyDatalog | Prolog |
|---|---|
| `+parent('bataa', 'bold')` | `parent(bataa, bold).` (атом — жижиг үсэг, хашилтгүй) |
| `father(X, Y) <= parent(X, Y) & male(X)` | `father(X, Y) :- parent(X, Y), male(X).` |
| `~has_child(X)` | `\+ has_child(X)` |
| `X != Y` | `X \= Y` |

Хувьсагч (X, Y, Z...) **том үсгээр** эхэлнэ, атом (bataa, bold...) **жижиг үсгээр**. Өгүүлбэр бүр цэгээр төгсөнө. Ижил нэртэй предикатын бүх баримтыг нэг дор бичнэ (эс бөгөөс `discontiguous` анхааруулга өгнө).

SWI-Prolog локал суулгаагүй тул **swish.swi-prolog.org**-д доорх `family.pl`-ийн агуулгыг хуулж оруулна.

### SWISH-д ажиллуулах алхам

1. https://swish.swi-prolog.org нээнэ.
2. Зүүн талын editor-т `family.pl`-ийн бүх агуулгыг хуулж тавина.
3. Баруун талын query мөрөнд доорх асуултуудыг нэг нэгээр нь бичиж **Run** дарна (эсвэл `?- ` дараа бичээд Enter):

```
?- findall(P-C, parent(P, C), L), length(L, N).
% Хариу: N = 18.

?- ancestor(bataa, temuulen).
% Хариу: true.

?- childless(temuulen).
% Хариу: true.

?- childless(bataa).
% Хариу: false.

?- findall(Y, sibling(oyuna, Y), L), length(L, N).
% Хариу: N = 4  (bold, tomor хоёр хоёр удаа орсон — доор тайлбарлав)

?- findall(Y, sibling(oyuna, Y), L), sort(L, S), length(S, N).
% Хариу: N = 2
```

**Олон хариу авах:** `?- sibling(oyuna, Y).` гэж асуугаад дэлгэц дээрх `;` товч дарж дараагийн хариуг харна. Бүгдийг нэг дор авахын тулд `findall/3` ашиглана.

**pyDatalog-д яагаад давхардал гараагүй вэ?** pyDatalog нь Datalog загвар ашигладаг бөгөөд үр дүнг **олонлог (set)** хэлбэрээр хадгалдаг — ижил хариулт хэд хэдэн өөр замаар батлагдсан ч нэг л удаа гарна (Алхам 4-ийн `X != Y` туршилтаар үүнийг аль хэдийн харсан). Харин Prolog-ийн `findall/3` нь **жагсаалт (list)** авдаг — ижил хариу хэдэн өөр замаар батлагдсан тоогоороо олон удаа орно. Энд `sibling(oyuna, Y)` нь `oyuna`-ийн эцэг эх хоёул (bataa, saraa) `parent` баримттай тул `bold`, `tomor` тус бүр **хоёр удаа** (нэг удаа bataa-гаар, нэг удаа saraa-гаар) батлагдаж жагсаалтад орно → 4 элемент. `sort/2` давхардлыг хасдаг тул 2 элемент үлдэнэ.

**Шинэчлэлт:** SWI-Prolog-ийг энэ компьютер дээр `winget`-ээр суулгаж, дээрх бүх асуултыг жинхэнэ `swipl`-аар ажиллуулж шалгасан — бүх хариу яг таамагласнаар гарсан (parent=18, ancestor(bataa,temuulen)=true, childless(temuulen)=true, childless(bataa)=false, sibling жагсаалт=4→sort хийвэл 2). Иймд та SWISH руу орохгүйгээр, терминалаас шууд ажиллуулж болно:

```
swipl family.pl
?- ancestor(bataa, temuulen).
```

эсвэл PowerShell-ээс шууд:

```
& "C:\Program Files\swipl\bin\swipl.exe" family.pl
```

## Нэмэлт даалгавар: Нөхөр эхнэр ба хуримын ураг

Мэдлэгийн санд "нөхөр эхнэр" гэсэн баримт шууд байхгүй — гэхдээ хэрэв хоёр хүн нэг хүүхдийн эцэг эх бол тэднийг хос гэж дүгнэж болно.

1. `spouse(X, Y)` — X, Y нь нэг хүүхдийн эцэг эх, мөн өөр хүн.
2. `uncle_in_law(X, Y)` — X нь Y-ийн авга эгчийн нөхөр.
3. `aunt_in_law(X, Y)` — X нь Y-ийн авга ахын эхнэр.

In [21]:
_ = pyDatalog.create_terms('spouse, uncle_in_law, aunt_in_law')

# spouse(X, Y): X, Y нэг хүүхдийн (Z) эцэг эх, мөн өөр хүн
spouse(X, Y) <= parent(X, Z) & parent(Y, Z) & (X != Y)

# uncle_in_law(X, Y): X нь Y-ийн авга эгчийн (цусан төрлийн aunt) нөхөр
uncle_in_law(X, Y) <= aunt(Z, Y) & spouse(X, Z) & male(X)

# aunt_in_law(X, Y): X нь Y-ийн авга ахын (цусан төрлийн uncle) эхнэр
aunt_in_law(X, Y) <= uncle(Z, Y) & spouse(X, Z) & female(X)

print('rules defined')

rules defined


**Шалгах:** `count("spouse(X, Y)")` нь 10 (5 хос, хоёр чиглэлд). `uncle_in_law(X, 'anar')`-аас dorj гарна, мөн dorj нь enkhee, zolboo-ийн uncle_in_law. `aunt_in_law` нийт 7 мөр гаргах ёстой.

In [22]:
assert count("spouse(X, Y)") == 10

print('uncle_in_law(X, anar):', uncle_in_law(X, 'anar'))
print('uncle_in_law(dorj, Y):', uncle_in_law('dorj', Y))

assert count("aunt_in_law(X, Y)") == 7
print('aunt_in_law(X, Y):')
print(aunt_in_law(X, Y))

print('all checks passed')

uncle_in_law(X, anar): X   
----
dorj
uncle_in_law(dorj, Y): Y     
------
enkhee
anar  
zolboo
aunt_in_law(X, Y):
X       | Y     
--------|-------
tsetseg | enkhee
tsetseg | munkh 
tsetseg | naraa 
ganaa   | naraa 
ganaa   | munkh 
ganaa   | anar  
ganaa   | zolboo
all checks passed


**4. Яагаад `spouse` дүрэм нь хүүхэдгүй хосыг илрүүлж чадахгүй вэ?** `spouse(X, Y)` нь зөвхөн `parent(X, Z) & parent(Y, Z)` — өөрөөр хэлбэл X, Y хоёул нэг хүүхдийн эцэг эх байх шаардлагатай гэдгээр баталгаажуулдаг. Хэрэв хос хүүхэдгүй бол ямар ч Z байхгүй тул энэ нөхцөл хэзээ ч биелэхгүй — **гэрлэлт бол `parent`-аас гаргаж авсан дериватив ойлголт, өөрөө анхдагч баримт биш** учраас. Үүнийг засахын тулд мэдлэгийн санд шууд `married(X, Y)` гэсэн **шинэ анхдагч баримт** нэмэх хэрэгтэй (жишээ нь `+married('bold', 'tsetseg')`), parent-ээс үл хамаарсан.

## Нэмэлт даалгавар: Шинэ гишүүн ба логик томъёо

Мэдлэгийн санд шинэ хүн нэмээд, **дүрмийг огт өөрчлөхгүйгээр** хариулт хэрхэн автоматаар шинэчлэгдэхийг ажиглая. `dulguun` нь bat, anar хоёрын хүүхэд (temuulen-ийн дүү/ах) болго.

In [23]:
+male('dulguun')
+parent('bat', 'dulguun')
+parent('anar', 'dulguun')

print('sibling(X, temuulen):', sibling(X, 'temuulen'))
print('ancestor(X, dulguun):', ancestor(X, 'dulguun'))
print('ancestor(X, temuulen):', ancestor(X, 'temuulen'))

assert count("sibling(X, 'temuulen')") == 1  # dulguun
assert count("ancestor(X, 'dulguun')") == count("ancestor(X, 'temuulen')") == 6
print('all checks passed')

sibling(X, temuulen):

 X      
-------
dulguun
ancestor(X, dulguun): X      
-------
bold   
tsetseg
bataa  
saraa  
bat    
anar   
ancestor(X, temuulen): X      
-------
bold   
tsetseg
bataa  
saraa  
anar   
bat    
all checks passed


**2. Дүрмүүдийг FOL томъёо болгов:**

- `father(X, Y) <= parent(X, Y) & male(X)`
  → ∀x ∀y (parent(x, y) ∧ male(x) → father(x, y))
  — X, Y хоёул дүрмийн толгойд байгаа тул **∀** (бүх x, y-д хамаарна).

- `grandparent(X, Y) <= parent(X, Z) & parent(Z, Y)`
  → ∀x ∀y (**∃z** (parent(x, z) ∧ parent(z, y)) → grandparent(x, y))
  — Z зөвхөн дүрмийн бие дотор байгаа тул **∃** (ядаж нэг ийм завсрын хүн олдоход хангалттай).

- `ancestor` (суурь + рекурсив хоёулаа):
  → ∀x ∀y (parent(x, y) → ancestor(x, y))
  → ∀x ∀y (**∃z** (parent(x, z) ∧ ancestor(z, y)) → ancestor(x, y))
  — Энд ч мөн Z нь ∃, учир нь зөвхөн дүрмийн биед л орсон.

**Ерөнхий дүрэм:** дүрмийн толгойд орсон хувьсагч → ∀ (гаднаас өгөгдөнө). Зөвхөн биед орсон хувьсагч → ∃ (дүрэм дотроо "ямар нэг X олдвол" гэж шийднэ).

**3. "Хүн бүр эцэг эхтэй":** ∀x ∃y parent(y, x).

Энэ мэдлэгийн санд **худал** — учир нь `bataa`-д `parent(Y, bataa)` гэсэн ямар ч баримт алга, мөн ямар ч дүрмээр үүнийг гаргаж авах боломжгүй. **Closed-world assumption**-ийн дагуу "баталгаажаагүй = худал" гэж үздэг тул систем "bataa-д эцэг эх алга" гэж хариулна — бодит амьдрал дээр bataa-д мэдээж эцэг эх байсан ч, **мэдлэгийн сан дутуу** учраас систем үүнийг мэдэхгүй. Энэ бол FOL мэдлэгийн сангийн чухал хязгаарлалт: систем зөвхөн **өөрт байгаа мэдээллээрээ** дүгнэлт гаргадаг, бодит ертөнцийн бүрэн үнэнээр биш.

In [24]:
# bataa-д эцэг эх байхгүй гэдгийг баталгаажуулъя
assert count("parent(Y, 'bataa')") == 0
print("parent(Y, 'bataa') олдсон тоо:", count("parent(Y, 'bataa')"), '-> CWA-ийн дагуу худал')

parent(Y, 'bataa') олдсон тоо: 0 -> CWA-ийн дагуу худал


## Хэлэлцүүлэг (4.11)

**1. Лаб 3-ын CSP-тэй харьцуулахад:** CSP нь нэг тодорхой бодлого (хувьсагч/домэйн/хязгаарлалт)-ыг шийдэхэд зориулагдсан — шийдэл олдсоны дараа түүнийг дахин бусад асуултад ашиглахгүй. Харин энэ лабын FOL мэдлэгийн сан бол **нэг тогтвортой мэдээллийн бааз** бөгөөд дээр нь дурын тооны өөр өөр query (parent, sibling, ancestor, custom...) тавьж болно — мэдлэгийн санг нэг удаа байгуулаад, дараа нь хэдэн ч өөр асуулт асууж болдог нь гол давуу тал.

**2. `grandparent`-ийг дүрэмгүйгээр баримтаар бичвэл 12 мөр болно** (12 grandparent хос тус бүрд нэг баримт). Шинэ хүн нэмэхэд **дүрэм** хамаагүй илүү хэмнэлттэй — `parent` баримт 1-2 мөр нэмэхэд `father`, `grandparent`, `ancestor`, `cousin`... гэх мэт бүх дүрэм автоматаар шинэчлэгдэнэ (яг л §4.9-д dulguun нэмэхэд харсан шиг). Баримтаар бол grandparent, uncle, cousin гэх мэт **бүх** шинэ холбоог гараар дахин тооцож нэмэх хэрэгтэй болно.

**3. CWA ба munkh:** Хэрэв munkh-ийн хүүхдийн мэдээлэл мэдлэгийн санд дутуу байвал (бодит дээр хүүхэдтэй ч бол) `childless('munkh')` **true** гэж буцаана — учир нь систем "баталгаажаагүй бүхэн худал" гэж үздэг. Энэ хариулт **систем хувьдаа зөв боловч бодит байдалтай зөрүүтэй** байж болно; FOL/Datalog-ийн дүгнэлт зөвхөн мэдлэгийн санд байгаа мэдээллээр хязгаарлагддаг гэдгийг энэ харуулж байна.

**4. Prolog ба pyDatalog-ийн ялгаа:** `ancestor` жишээн дээр хоёул зөв, ижил хариу өгсөн — учир нь recursion-ий логик ижил. Харин `sibling`-д ялгаа гарсан: **pyDatalog** (Datalog, backward chaining боловч үр дүнг **олонлог** болгож хадгалдаг) нь ижил хариуг нэг л удаа өгдөг (sibling(oyuna,Y) → 2 утга), харин **Prolog** (depth-first хайлт, `findall` бүх баталгааны замыг **жагсаалтад** цуглуулдаг) ижил хариуг хэд хэдэн удаа (sibling → 4 утга, cousin → 32) буцаана, учир нь өөр өөр Z (эсвэл дунд шатны хүн)-ээр дамжиж ижил дүгнэлтэд хүрэх боломжтой бол тэр бүрийг тусад нь тоолдог.

`uncle`/`aunt` дүрэм зөвхөн **цусан төрлийн** авгыг хамардаг — хуримаар орсон авга (жишээ нь dorj) үүгээр олдохгүй. Бид үүнийг §4.8-д тусад нь `uncle_in_law`/`aunt_in_law` дүрмээр нэмж засав, өөрөөр хэлбэл анхны дүрэм **бүх** тохиолдлыг хамарсангүй — энэ бол мэдлэгийн сан зохион байгуулахдаа ямар ойлголтыг нэг дүрэмд оруулахаа сайн тодорхойлох хэрэгтэйг харуулна.

## Дүгнэлт

Энэ лабораторид нэг жижиг баримтын сан дээр дүрэм бичиж, шинэ холбоог (өвөө эмээ, ах дүү, үеэл, өвөг дээдэс) **программд тусад нь зааглгүйгээр** өөрөө гаргаж авахыг харлаа. Мөн нэг мэдлэгийн санг pyDatalog ба Prolog хоёр хэлээр бичиж, тэдгээрийн **олонлог vs жагсаалт** семантикийн ялгаа рекурсив дүрэм болон үгүйсгэлд хэрхэн нөлөөлдгийг бодитоор ажиглав. Unification, дүрэм, рекурсив, closed-world assumption гэсэн ойлголтууд логик дүгнэлтийн дараагийн сэдвүүдийн үндэс болно.